# File handling and biological formats
## 00-10 From functions to files
Session 06 named repeated calculations. Now inputs live in separate files and results persist after Python exits. A file is named stored text; a directory groups files. A FASTA record stores a sequence, not a biological finding. Our invented examples are one represented DNA strand, using only A/T/G/C. [Dataset descriptions](../../datasets/README.md) define every file.

Input -> reader -> validation -> counts and GC -> CSV report

Today adds paths, file closing and predictable failure handling. No previous notebook must run.

## 10-25 Paths
`pathlib` is a standard-library module; `Path` represents a path. `Path.cwd()` is the process working directory. A relative path starts there; an absolute path starts at a drive/root. Spaces are valid inside a path; quote command-line paths. `/` joins Path components, not numerical division here. This notebook expects its own session folder OR the course root as working directory. The next cell checks both and reports a missing extraction clearly without using exception handling before we teach it.

`exists()` tests presence; `is_absolute()` tests path form; `resolve()` makes an absolute path. `parents[2]` is the third enclosing folder. Expected: Dataset folder found: True. Task 1: predict where Path('outputs') points.

In [ ]:
from pathlib import Path

course_root = Path.cwd()
if not (course_root / "datasets").is_dir():
    course_root = Path.cwd().parents[1]
data_folder = course_root / "datasets" / "session-07"
print("Dataset folder found:", data_folder.is_dir())
print("Relative path:", Path("outputs").is_absolute())
print("Working folder:", Path.cwd().name)

## 25-45 Read text before CSV
`open` returns a file handle. `with` closes it when the block ends, including when an error occurs. Default mode reads; UTF-8 is the chosen text encoding. `read()` returns the whole tiny file as a string, including a final newline. Normalize case/end whitespace after reading. Expected ATGCAT, length6. Task 2: explain why read data and a Python string can use the same methods.

In [ ]:
with open(data_folder / "sequence.txt", encoding="utf-8") as handle:
    sequence = handle.read().strip().upper()
print(sequence)
print(len(sequence))

Writing uses mode `w`: it creates or overwrites the named file. Never write to your input file. Newline `\n` starts a new text line. Open [03_TEXT_CSV.py](03_TEXT_CSV.py) for the writing demonstration, run it from a writable working folder, and inspect outputs/practice.txt. It writes Synthetic DNA: ATGC. Its CSV demo writes a header then Toy,4. The notebook itself only reads data; executing it does not create reports.

CSV means comma-separated values. Rows are records; the first row here names columns. Use `csv`, not split(','): quoted commas need a real CSV reader. DictReader connects header names to strings. Numeric fields would need int/float conversion. Use newline='' for CSV reading/writing and encoding='utf-8'. Expected Toy_A ATGCAT and Toy_B GCGC. Task 3: write a separate two-column report using the script pattern.

In [ ]:
import csv
with open(data_folder / "sequences.csv", newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        print(row["id"], row["sequence"])

## 45-60 FASTA structure
FASTA stores sequence records with a header starting `>`, followed by one or more sequence lines. First header token is our ID; remaining words are a description. For example:

```text
>Toy_A synthetic educational record
atgc
AT
>Toy_B synthetic educational record
GC
GC
```

Join sequence lines within each header: Toy_A=ATGCAT; Toy_B=GCGC. CSV has columns; FASTA uses headers and possibly wrapped sequence lines. Task 4: mark headers and reconstruct each DNA string by hand.

Our small reader supports UTF-8 text, blank lines, stripped line ends, lower/upper DNA, unique nonempty first-token IDs and multiline records. It rejects sequence before a header, empty records/headers, duplicate IDs, internal spaces and all non-ATGC symbols including N. Descriptions are not retained. Blank lines are ignored deliberately, not sequence symbols. This is an educational subset, not a production FASTA parser; real formats may permit other alphabets or conventions. Biopython SeqIO comes in Session 10.

## 60-75 Expected failures
`try` encloses an operation that may fail. `except` handles only the named error. A missing path raises FileNotFoundError; an existing but empty file is different and our reader raises ValueError. `raise ValueError(message)` signals invalid content. `as error` gives the message. Bad numeric text also raises ValueError during conversion. Do not use bare except or suppress every error: it can hide a coding bug. Task 5: distinguish wrong path, empty file, ATN, and int('four'). Expected: Bad whole-number text: four.

In [ ]:
number_text = "four"
try:
    value = int(number_text)
    print(value)
except ValueError:
    print("Bad whole-number text:", number_text)

## 75-105 Build a checked file workflow
Problem: read all records and save ID, length, four counts, GC%. Algorithm: collect wrapped lines under IDs; reject malformed content; validate every complete record; loop to count; build rows; only then open the output. This prevents invalid records from producing a partial or overwritten report. It is not an atomic writer: an OS failure during writing can still interrupt a file.

`startswith` tests the header marker; `split()[0]` picks the first whitespace-separated token. `continue` skips documented blank lines. `writerow` writes one row; `writerows` writes a list of rows. Read the reader first, then summarize_records, then save_report in [file_tools.py](file_tools.py). The same functions are included below to keep this notebook self-contained. Functions retain Session 06's counting and GC algorithm; no cross-session import is needed.

In [ ]:
"""Limited educational FASTA reader and validated CSV writer; synthetic DNA only."""
import csv

### Build validate_dna

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def validate_dna(sequence):
    """Check already-normalized nonempty A/T/G/C text."""
    if sequence == "":
        return False
    for base in sequence:
        if base not in "ATGC":
            return False
    return True

### Build read_fasta

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def read_fasta(path):
    """Read unique first-token IDs and wrapped DNA; raise ValueError on bad data."""
    records = {}
    current_id = None
    with open(path, encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line == "":
                continue
            if line.startswith(">"):
                # Reject an empty previous record before moving to the next.
                if current_id is not None and records[current_id] == "":
                    raise ValueError("Empty sequence: " + current_id)
                header = line[1:].strip()
                if header == "":
                    raise ValueError("Empty FASTA header")
                current_id = header.split()[0]
                if current_id in records:
                    raise ValueError("Duplicate identifier: " + current_id)
                records[current_id] = ""
            else:
                if current_id is None:
                    raise ValueError("Sequence before first header")
                sequence_line = line.upper()
                if not validate_dna(sequence_line):
                    raise ValueError("Invalid DNA symbols: " + current_id)
                records[current_id] = records[current_id] + sequence_line
    if len(records) == 0:
        raise ValueError("No FASTA records")
    if records[current_id] == "":
        raise ValueError("Empty sequence: " + current_id)
    return records

### Build summarize_records

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def summarize_records(records):
    """Validate all records before output; return rows with counts and GC percent."""
    if len(records) == 0:
        raise ValueError("No records to report")
    rows = []
    for sequence_id in records:
        sequence = records[sequence_id]
        if sequence_id.strip() == "" or not validate_dna(sequence):
            raise ValueError("Invalid report record: " + sequence_id)
        counts = {"A": 0, "T": 0, "G": 0, "C": 0}
        for base in sequence:
            counts[base] = counts[base] + 1
        gc_percent = (counts["G"] + counts["C"]) / len(sequence) * 100
        rows.append([sequence_id, len(sequence), counts["A"], counts["T"],
                     counts["G"], counts["C"], gc_percent])
    return rows

### Build save_report

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def save_report(records, output_path):
    """Check every record before opening the output; overwrite only the named report."""
    rows = summarize_records(records)
    with open(output_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "length", "A", "T", "G", "C", "gc_percent"])
        writer.writerows(rows)

Run just the reader and summary first. Expected Toy_A length6 A2 T2 G1 C1 GC33.333...%; Toy_B length4 A0 T0 G2 C2 GC100%. Counts sum to length. Now open [02_DEMO.py](02_DEMO.py): `__file__` is the saved script's path, so input lookup is independent of cwd. The output folder is explicitly cwd/outputs. mkdir(exist_ok=True) creates that one folder if absent. Running from course root overwrites only outputs/demo_report.csv. Expected Saved records: 2 and Report: demo_report.csv. Task 6: run from a folder whose name contains spaces, then read the report with csv.DictReader and compare every row to the hand counts.

In [ ]:
records = read_fasta(data_folder / "sequences.fasta")
print(records)
for row in summarize_records(records):
    print(row)

## 105-115 Diagnose without data loss
This loop intentionally attempts missing, empty and invalid files. Each message identifies the failure type. Empty TXT is passed to the FASTA reader only as an empty-file test. No missing file fixture is created. Expected: Missing: missing.fasta; Invalid: empty.txt No FASTA records; Invalid: invalid.fasta Invalid DNA symbols: Bad. Task 7: also try duplicate.fasta and no_header.fasta; explain why rejecting them prevents silent record loss.

In [ ]:
for name in ["missing.fasta", "empty.txt", "invalid.fasta"]:
    try:
        print(read_fasta(data_folder / name))
    except FileNotFoundError:
        print("Missing:", name)
    except ValueError as error:
        print("Invalid:", name, error)

## 115-120 Summary and troubleshooting
A wrong cwd changes relative paths; __file__ anchors a saved script. Paths with spaces work when command arguments are quoted. FileNotFoundError: inspect path and extracted folders. ValueError: read the content message, do not discard the record. PermissionError: choose a writable output folder, close a report held by another application, then retry; we do not silently catch it here. A string read from CSV is not yet a number. Keep input/output paths distinct; `w` overwrites. Missing and empty are different states. Complete the assessment. Homework builds a new report without plotting or advanced parsing.